In [3]:
import pandas as pd
import os

# Percorsi dei file
raw_path = 'data/raw/'
proc_path = 'data/processed/'

# 1. Pulizia EDGES
print("Processing edges...")
edges = pd.read_csv(os.path.join(raw_path, 'edges.csv'))
# Rinominiamo le colonne Neo4j in standard Python
edges = edges.rename(columns={':START_ID': 'source', ':END_ID': 'target', ':TYPE': 'type'})
edges.to_csv(os.path.join(proc_path, 'edges_cleaned.csv'), index=False)

# 2. Pulizia NODES
print("Processing nodes...")
nodes = pd.read_csv(os.path.join(raw_path, 'nodes.csv'))

nodes = nodes.rename(columns={
    ':ID': 'id', 
    'celex_id': 'celex',
    'domains:STRING[]': 'domains',       
    'subdomains:STRING[]': 'subdomains', 
    'eurovoc_concepts:STRING[]': 'eurovoc_concepts'
})

# Teniamo solo le colonne utili per il primo task per non appesantire la memoria
cols_to_keep = ['id', 'celex', 'work_title', 'domains', 'subdomains', 'sector_label']
existing_cols = [c for c in cols_to_keep if c in nodes.columns]
nodes_cleaned = nodes[existing_cols]

nodes_cleaned.to_csv(os.path.join(proc_path, 'nodes_cleaned.csv'), index=False)

print(f"Pre-processing completato! Colonne salvate: {existing_cols}")

Processing edges...
Processing nodes...
Pre-processing completato! Colonne salvate: ['id', 'celex', 'work_title', 'domains', 'subdomains', 'sector_label']


In [1]:
import pandas as pd
import os

# Caricamento
proc_path = os.path.join('..', 'data', 'processed')
nodes = pd.read_csv(os.path.join(proc_path, 'nodes_cleaned.csv'))

# Pulizia: Sostituiamo i NaN con etichette leggibili
nodes['work_title'] = nodes['work_title'].fillna('Titolo non disponibile')
nodes['domains'] = nodes['domains'].fillna('00 UNKNOWN') # Usiamo 00 per i domini mancanti
nodes['celex'] = nodes['celex'].fillna('N/A')

# Salvataggio sovrascrivendo il vecchio file
nodes.to_csv(os.path.join(proc_path, 'nodes_cleaned.csv'), index=False)
print("NaN gestiti con successo in nodes_cleaned.csv")

NaN gestiti con successo in nodes_cleaned.csv


In [3]:
import pandas as pd
import os

# 1. Definizione percorsi (usiamo i percorsi relativi alla radice del progetto)
raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

def clean_headers(df):
    """Rimuove i suffissi Neo4j dai nomi delle colonne (es. id:ID -> id)"""
    new_cols = []
    for col in df.columns:
        clean_name = col.split(':')[0] if ':' in col else col
        # Gestisce i casi tipo :LABEL o :TYPE (rimuove il due punti iniziale)
        if not clean_name and ':' in col:
            clean_name = col.split(':')[1]
        new_cols.append(clean_name.lower())
    df.columns = new_cols
    return df

print("--- Inizio Normalizzazione Dataset ---")

# --- A. Pulizia NODES (Gestione NaN definitiva) ---
nodes_file = os.path.join(proc_path, 'nodes_cleaned.csv')
if os.path.exists(nodes_file):
    nodes = pd.read_csv(nodes_file)
    nodes['work_title'] = nodes['work_title'].fillna('Titolo non disponibile')
    nodes['domains'] = nodes['domains'].fillna('00 UNKNOWN')
    nodes['celex'] = nodes['celex'].fillna('N/A')
    nodes.to_csv(nodes_file, index=False)
    print(f"1. nodes_cleaned.csv: NaN risolti per {len(nodes)} righe.")

# --- B. Pulizia EUROVOC CONCEPTS (Dizionario) ---
concepts_raw = os.path.join(raw_path, 'eurovoc_concept.csv')
if os.path.exists(concepts_raw):
    concepts = pd.read_csv(concepts_raw)
    concepts = clean_headers(concepts)
    # Rimuoviamo le parentesi quadre STRING[] dai dati se presenti
    for col in ['eurovoc_concepts', 'domains', 'subdomains']:
        if col in concepts.columns:
            concepts[col] = concepts[col].str.replace(r'[\[\]"]', '', regex=True)
    concepts.to_csv(os.path.join(proc_path, 'eurovoc_concepts_cleaned.csv'), index=False)
    print("2. eurovoc_concepts_cleaned.csv: Creato (nomi colonne puliti).")

# --- C. Pulizia ARCHI (has_concept e parent_of) ---
for f_name in ['has_concept_edges.csv', 'parent_of.csv']:
    f_path = os.path.join(raw_path, f_name)
    if os.path.exists(f_path):
        df = pd.read_csv(f_path)
        # Rinominiamo i :START_ID e :END_ID in source e target
        df.columns = ['source', 'target', 'type']
        out_name = f_name.replace('.csv', '_cleaned.csv')
        df.to_csv(os.path.join(proc_path, out_name), index=False)
        print(f"3. {out_name}: Creato con colonne standard (source, target).")

print("--- Normalizzazione Completata! ---")

--- Inizio Normalizzazione Dataset ---
1. nodes_cleaned.csv: NaN risolti per 88130 righe.
2. eurovoc_concepts_cleaned.csv: Creato (nomi colonne puliti).
3. has_concept_edges_cleaned.csv: Creato con colonne standard (source, target).
3. parent_of_cleaned.csv: Creato con colonne standard (source, target).
--- Normalizzazione Completata! ---
